In [ ]:
import math

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
import sympy as sym

from rotation.symbolic import ALL_SYMBOL, create_3d_rotation, create_mix_equation, create_kinetic, create_axial_mix_equation
from rotation.solver import initialize_system, create_func, solve_mixing

In [ ]:
# Mass info from online PDG (in GeV)
# https://pdglive.lbl.gov/Particle.action?node=S009
# https://pdglive.lbl.gov/Particle.action?node=S014
PDG = {
    ALL_SYMBOL["prop"]["pi0"]: 0.1349768,
    ALL_SYMBOL["prop"]["eta"]: 0.547862,
}

In [ ]:
def format_output(key, value):
    """Helps format solutions for printing"""
    if key in [ALL_SYMBOL["prop"]["pi0"], ALL_SYMBOL["bare"]["pi0"]]:
        value *= sym.Symbol(r" m_{\pi^0\text{, PDG}}") / PDG[ALL_SYMBOL["prop"]["pi0"]]
    elif key in [ALL_SYMBOL["prop"]["eta"], ALL_SYMBOL["bare"]["eta"]]:
        value *= sym.Symbol(r" m_{\eta\text{, PDG}}") / PDG[ALL_SYMBOL["prop"]["eta"]]
    elif key in [ALL_SYMBOL["prop"]["phi"], ALL_SYMBOL["prop"]["eta0"], ALL_SYMBOL["bare"]["phi"], ALL_SYMBOL["bare"]["pi0-phi"], ALL_SYMBOL["bare"]["eta-phi"], ALL_SYMBOL["bare"]["eta0"]]:
        value *= sym.Symbol(r" \text{ GeV}")
    elif key in ALL_SYMBOL["angle"].values():
        value *= sym.Symbol(r" \pi") / math.pi
    return value

In [ ]:
rotate = create_3d_rotation()[0]
equation = create_mix_equation(rotate)[0]
argument_order, system_of_equations = initialize_system(equation.lhs - equation.rhs)

In [ ]:
# Fixed known parameters 
known = {
    ALL_SYMBOL["prop"]["pi0"]: PDG[ALL_SYMBOL["prop"]["pi0"]],
    ALL_SYMBOL["prop"]["eta"]: PDG[ALL_SYMBOL["prop"]["eta"]],
    ALL_SYMBOL["bare"]["phi"]: 1.0,
    ALL_SYMBOL["bare"]["pi0-phi"]: 0.1,
    ALL_SYMBOL["bare"]["eta-phi"]: 0.3,
}

# These will be the unknowns we numerically solve for (the values are
# just initial guesses to get the solver going)
# Remember: the system of equations can only solve for up to 6 unknowns!
unknown = {
    ALL_SYMBOL["prop"]["phi"]: 1.0,
    ALL_SYMBOL["bare"]["pi0"]: PDG[ALL_SYMBOL["prop"]["pi0"]],
    ALL_SYMBOL["bare"]["eta"]: PDG[ALL_SYMBOL["prop"]["eta"]],
    ALL_SYMBOL["angle"]["alpha"]: 0.0,
    ALL_SYMBOL["angle"]["beta"]: 0.0,
    ALL_SYMBOL["angle"]["gamma"]: 0.0,
}

In [ ]:
# Create function
func = create_func(
    system_of_equations,
    argument_order,
    known,
)
try:
    # Solve system
    solved = solve_mixing(
        func,
        argument_order,
        unknown,
    )

    # Printout the system with our known values, and unknowns blank
    print("Arguments input:")
    for k in argument_order:
        if k in known:
            sym.Eq(k, known[k], evaluate=False)
        else:
            sym.Eq(k, sym.Symbol("?"), evaluate=False)
    equation.subs(known)

    # Printout the system with our solved unknowns, and knowns blank
    print("Arguments solved:")
    for k, v in solved.items():
        v = format_output(k, v)
        sym.Eq(k, v, evaluate=False)

    print("The solved equation:")
    equation.subs(solved)

    print("Compare LHS and RHS evaluations:")
    sym.Eq(
        (1*equation.lhs.subs(known|solved)).evalf(),
        (1*equation.rhs.subs(known|solved)).evalf(),
        evaluate=False
    )

except ValueError as e:
    print(e)
    print("The solver failed! Try other inputs.")

In [ ]:
axial_eqn = create_axial_mix_equation(rotate, create_kinetic())[0]
axial_order, axial_system = initialize_system(axial_eqn.lhs - axial_eqn.rhs)

In [ ]:
# Fixed known parameters 
axial_known = {
    ALL_SYMBOL["prop"]["pi0"]: PDG[ALL_SYMBOL["prop"]["pi0"]],
    ALL_SYMBOL["prop"]["eta"]: PDG[ALL_SYMBOL["prop"]["eta"]],
    ALL_SYMBOL["prop"]["eta0"]: PDG[ALL_SYMBOL["prop"]["eta"]]*0.9,
    ALL_SYMBOL["angle"]["u"]: 0.5,
    ALL_SYMBOL["angle"]["v"]: 0,
    ALL_SYMBOL["angle"]["w"]: 0,
}

# These will be the unknowns we numerically solve for (the values are
# just initial guesses to get the solver going)
# Remember: the system of equations can only solve for up to 6 unknowns!
axial_unknown = {
    ALL_SYMBOL["bare"]["pi0"]: PDG[ALL_SYMBOL["prop"]["pi0"]],
    ALL_SYMBOL["bare"]["eta"]: PDG[ALL_SYMBOL["prop"]["eta"]],
    ALL_SYMBOL["bare"]["eta0"]: PDG[ALL_SYMBOL["prop"]["eta"]]*0.9,
    ALL_SYMBOL["angle"]["alpha"]: 0.0,
    ALL_SYMBOL["angle"]["beta"]: 0.0,
    ALL_SYMBOL["angle"]["gamma"]: 0.0,
}

In [ ]:
# Create function
func = create_func(
    axial_system,
    axial_order,
    axial_known,
)
try:
    # Solve system
    axial_solved = solve_mixing(
        func,
        axial_order,
        axial_unknown,
    )

    # Printout the system with our known values, and unknowns blank
    print("Arguments input:")
    for k in axial_order:
        if k in axial_known:
            sym.Eq(k, axial_known[k], evaluate=False)
        else:
            sym.Eq(k, sym.Symbol("?"), evaluate=False)
    axial_eqn.subs(axial_known)

    # Printout the system with our solved unknowns, and knowns blank
    print("Arguments solved:")
    for k, v in axial_solved.items():
        v = format_output(k, v)
        sym.Eq(k, v, evaluate=False)

    print("The solved equation:")
    axial_eqn.subs(axial_solved)

    print("Compare LHS and RHS evaluations:")
    sym.Eq(
        (1*axial_eqn.lhs.subs(axial_known|axial_solved)).evalf(),
        (1*axial_eqn.rhs.subs(axial_known|axial_solved)).evalf(),
        evaluate=False
    )

except ValueError as e:
    print(e)
    print("The solver failed! Try other inputs.")